# 🎙️ PySpark — Speech Emotion Recognition Pipeline

**Pipeline industrialisé avec PySpark** inspiré du notebook Conv1D de Dmitry Babko,
corrigé pour éliminer le **Data Leakage** et les **OutOfMemoryError**.

### Étapes du pipeline :
1. **Imports & Session Spark**
2. **Chargement des fichiers audio** (`binaryFile`)
3. **Extraction des labels** depuis les noms de fichiers
4. **Split anti-leakage** (80/20 AVANT toute manipulation) + **Micro-Partitionnement**
5. **Data Augmentation** (Train uniquement) — Pitch Shift, Bruit Blanc, Time Stretch
6. **Extraction de features multi-échelle** (185 dimensions — 9 caractéristiques Babko)
7. **Sauvegarde en Parquet** (sans `.cache()` pour éviter l'OOM)
8. **Vérification audio** (écoute original vs augmenté)

### Corrections vs notebook original Babko :
- ✅ Split **AVANT** augmentation (anti-leakage)
- ✅ Pas de `.cache()` sur les binaires audio (anti-OOM)
- ✅ Stratégie **Write-then-Read** Parquet pour libérer la RAM JVM
- ✅ UDFs avec `sr` hardcodé (pas de closure sur `SAMPLE_RATE`)
- ✅ 9 features Babko complètes (185 dimensions)

### Leviers anti-OOM ajoutés :
- 🔧 **Micro-Partitionnement** : `.repartition(200)` après le split
- 🔧 **Persistance Itérative** : écriture séquentielle `overwrite` + `append` au lieu de `unionByName`

---
## 📦 Étape 1 — Imports & Session Spark

In [1]:
import os
import io
import numpy as np
import librosa
import soundfile as sf

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    udf, col, lit, monotonically_increasing_id
)
from pyspark.sql.types import (
    BinaryType, ArrayType, FloatType, StringType
)

# --- Création de la session Spark ---
# ⚠️ Mémoire augmentée pour supporter l'audio binaire
# Arrow activé pour les transferts JVM ↔ Python rapides
spark = (
    SparkSession.builder
    .appName("SER_Pipeline")
    .master("local[4]")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "4g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.driver.maxResultSize", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# Constante utilisée côté driver uniquement (PAS dans les UDFs)
SAMPLE_RATE = 22050

print("✅ Session Spark démarrée :", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 20:16:03 WARN Utils: Your hostname, Joane-PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/08 20:16:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 20:16:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Session Spark démarrée : 4.1.1


---
## 📂 Étape 2 — Chargement des fichiers audio (`binaryFile`)

In [2]:
# 📁 Chemin vers le dataset — WSL2 monte le disque Windows sous /mnt/c/
DATASET_PATH = "/mnt/c/Users/joane/Desktop/ESGI4/Spark Core/voice-rec/Dataset/"

# Chargement de tous les fichiers .wav en binaire
# binaryFile lit le contenu brut de chaque fichier dans la colonne "content"
df_raw = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.wav")
    .option("recursiveFileLookup", "true")
    .load(DATASET_PATH)
    .select(
        col("path"),
        col("content")
    )
)

print(f"📂 Nombre total de fichiers audio chargés : {df_raw.count()}")
df_raw.printSchema()

📂 Nombre total de fichiers audio chargés : 12162
root
 |-- path: string (nullable = true)
 |-- content: binary (nullable = true)



---
## 🏷️ Étape 3 — Extraction des Labels depuis les noms de fichiers

Datasets supportés :
- **RAVDESS** : `03-01-05-01-02-01-24.wav` → 3ème segment = émotion
- **CREMA-D** : `1001_DFA_ANG_XX.wav` → 3ème segment = code émotion
- **TESS** : `OAF_back_angry.wav` → dernier segment = émotion
- **SAVEE** : `DC_a01.wav` → lettres dans le 2ème segment = émotion

In [4]:
@udf(returnType=StringType())
def extract_label(path):
    """Extrait le label d'émotion à partir du chemin du fichier audio.
    Supporte RAVDESS, CREMA-D, TESS et SAVEE."""
    import os
    filename = os.path.basename(path).lower()

    # --- RAVDESS : ex. "03-01-05-01-02-01-24.wav" ---
    # Le 3ème champ (index 2) encode l'émotion
    if filename.startswith("03-01") or filename.count("-") >= 5:
        parts = filename.replace(".wav", "").split("-")
        if len(parts) >= 3:
            ravdess_map = {
                "01": "neutral", "02": "neutral",
                "03": "happy",   "04": "sad",
                "05": "angry",   "06": "fear",
                "07": "disgust", "08": "surprise"
            }
            return ravdess_map.get(parts[2], "unknown")

    # --- CREMA-D / TESS ---
    if "_" in filename:
        parts = filename.replace(".wav", "").split("_")

        # TESS : "OAF_back_angry.wav" → parts[2] = émotion
        if len(parts) == 3 and parts[2] in (
            "angry", "disgust", "fear", "happy", "neutral",
            "sad", "surprise", "ps"
        ):
            return parts[2]

        # CREMA-D : "1001_DFA_ANG_XX.wav" → parts[2] = code émotion
        if len(parts) >= 3:
            crema_map = {
                "ANG": "angry", "DIS": "disgust", "FEA": "fear",
                "HAP": "happy", "NEU": "neutral", "SAD": "sad"
            }
            emotion_code = parts[2].upper()
            return crema_map.get(emotion_code, "unknown")

    # --- SAVEE : ex. "DC_a01.wav" ---
    if "_" in filename:
        savee_name = filename.replace(".wav", "")
        emotion_part = savee_name.split("_")[-1]
        emotion_letters = "".join(c for c in emotion_part if c.isalpha())
        savee_map = {
            "a": "angry", "d": "disgust", "f": "fear",
            "h": "happy", "n": "neutral", "sa": "sad",
            "su": "surprise"
        }
        return savee_map.get(emotion_letters, "unknown")

    return "unknown"


# Application de l'UDF sur le DataFrame brut
df_labeled = df_raw.withColumn("label", extract_label(col("path")))

# Filtrer les labels inconnus
df_labeled = df_labeled.filter(col("label") != "unknown")

print(f"🏷️  Échantillons labellisés : {df_labeled.count()}")
df_labeled.groupBy("label").count().orderBy("count", ascending=False).show()

🏷️  Échantillons labellisés : 12162


+--------+-----+
|   label|count|
+--------+-----+
|   angry| 1923|
|     sad| 1923|
|    fear| 1923|
|   happy| 1923|
| disgust| 1923|
| neutral| 1895|
|      ps|  400|
|surprise|  252|
+--------+-----+



---
## 🔀 Étape 4 — Split Anti-Leakage (80/20) + Micro-Partitionnement

> ⚠️ **Le split est effectué AVANT toute augmentation** pour éviter la fuite de données
> observée dans le notebook original de Babko.
>
> Si on augmente d'abord puis on split, des versions transformées du même fichier
> peuvent se retrouver à la fois dans le train et le test → **scores gonflés artificiellement**.

> 💡 **Levier 1 — Micro-Partitionnement** : après le split, on applique `.repartition(200)` sur le
> train pour réduire le volume de données binaires chargées par tâche Spark et
> éviter de saturer le Heap Space JVM.

In [5]:
# Split AVANT toute manipulation = anti-leakage
df_train, df_test = df_labeled.randomSplit([0.8, 0.2], seed=42)

# --- Levier 1 : Micro-Partitionnement ---
# Repartitionner le train en 200 partitions pour réduire le volume
# de données binaires chargées par tâche Spark (anti-OOM)
df_train = df_train.repartition(200)

# ⚠️ PAS de .cache() ici — les binaires audio saturent le Heap Space JVM

train_count = df_train.count()
test_count = df_test.count()
print(f"🔀 Split anti-leakage + micro-partitionnement :")
print(f"   Train = {train_count} échantillons (repartitionné en 200 partitions)")
print(f"   Test  = {test_count} échantillons")

# --- Vérification : aucun fichier en commun ---
train_paths = set(df_train.select("path").rdd.flatMap(lambda x: x).collect())
test_paths = set(df_test.select("path").rdd.flatMap(lambda x: x).collect())
overlap = train_paths & test_paths
assert len(overlap) == 0, f"❌ FUITE DE DONNÉES : {len(overlap)} fichiers dans les deux sets !"
print("✅ Aucune fuite de données — train et test sont disjoints.")

🔀 Split anti-leakage + micro-partitionnement :
   Train = 9680 échantillons (repartitionné en 200 partitions)
   Test  = 2482 échantillons


✅ Aucune fuite de données — train et test sont disjoints.


---
## 🔧 Étape 5 — Data Augmentation (Train uniquement)

3 transformations appliquées via des UDFs PySpark :

| Augmentation | Librairie | Paramètres |
|---|---|---|
| **Pitch Shifting** | `librosa.effects.pitch_shift` | `n_steps=4` |
| **Bruit Blanc** | `np.random.randn` | `noise_factor=0.005` |
| **Time Stretching** | `librosa.effects.time_stretch` | `rate=1.2` |

Chaque UDF : `binary audio → transformation → binary audio` via `io.BytesIO`.

> ⚠️ Le `sr=22050` est **hardcodé** dans chaque UDF pour éviter les problèmes
> de sérialisation de closure PySpark.

> 💡 **Levier 2 — Persistance Itérative** : au lieu de `unionByName` des 4 DataFrames
> en mémoire, chaque version est écrite séquentiellement en Parquet (`overwrite` puis
> `append`), avec `spark.catalog.clearCache()` entre chaque écriture pour forcer
> la JVM à libérer les objets binaires.

In [6]:
@udf(returnType=BinaryType())
def pitch_shift_udf(audio_bytes):
    """Pitch shifting (+4 demi-tons).
    Entrée : bytes WAV bruts  →  Sortie : bytes WAV transformés."""
    import io, librosa, numpy as np, soundfile as sf
    try:
        # Décodage du binaire WAV en signal numpy
        y, sr = librosa.load(io.BytesIO(audio_bytes), sr=22050)
        # Décalage de 4 demi-tons vers le haut
        y_shifted = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=4)
        # Ré-encodage en binaire WAV
        buffer = io.BytesIO()
        sf.write(buffer, y_shifted, sr, format="WAV")
        return buffer.getvalue()
    except Exception:
        return audio_bytes  # Fallback : retourner l'original


@udf(returnType=BinaryType())
def add_noise_udf(audio_bytes):
    """Ajout de bruit blanc gaussien (facteur 0.005).
    Entrée : bytes WAV bruts  →  Sortie : bytes WAV bruités."""
    import io, librosa, numpy as np, soundfile as sf
    try:
        y, sr = librosa.load(io.BytesIO(audio_bytes), sr=22050)
        # Génération du bruit blanc
        noise = np.random.randn(len(y)) * 0.005
        y_noisy = y + noise.astype(np.float32)
        buffer = io.BytesIO()
        sf.write(buffer, y_noisy, sr, format="WAV")
        return buffer.getvalue()
    except Exception:
        return audio_bytes


@udf(returnType=BinaryType())
def time_stretch_udf(audio_bytes):
    """Time stretching (facteur 1.2 = 20% plus rapide).
    Entrée : bytes WAV bruts  →  Sortie : bytes WAV étirés."""
    import io, librosa, numpy as np, soundfile as sf
    try:
        y, sr = librosa.load(io.BytesIO(audio_bytes), sr=22050)
        # Accélération de 20% sans changer le pitch
        y_stretched = librosa.effects.time_stretch(y=y, rate=1.2)
        buffer = io.BytesIO()
        sf.write(buffer, y_stretched, sr, format="WAV")
        return buffer.getvalue()
    except Exception:
        return audio_bytes


print("🔧 UDFs d'augmentation définies — prêtes à être appliquées.")

🔧 UDFs d'augmentation définies — prêtes à être appliquées.


In [7]:
# =================================================================
# Levier 2 : Persistance Itérative (Checkpointing Manuel)
# -----------------------------------------------------------------
# Au lieu de unionByName (4 DataFrames binaires en RAM = OOM),
# on écrit chaque version séquentiellement en Parquet :
#   1. original    → mode overwrite  (crée le fichier)
#   2. pitch_shift → mode append
#   3. white_noise → mode append
#   4. time_stretch → mode append
# Avec clearCache() entre chaque étape pour libérer le Heap JVM.
# =================================================================

AUGMENTED_PARQUET = "/mnt/c/Users/joane/Desktop/ESGI4/Spark Core/voice-rec/augmented_train.parquet"

# Schéma commun : path, content, label, augmentation
# Toutes les écritures suivent ce schéma pour garantir la cohérence.

# --- 1/4 : Originaux (overwrite = crée le Parquet) ---
print("🔧 [1/4] Écriture des originaux (overwrite)...")
df_train_original = df_train.withColumn("augmentation", lit("original"))
df_train_original.write.mode("overwrite").parquet(AUGMENTED_PARQUET)
print(f"   ✅ Originaux sauvegardés ({train_count} fichiers)")

# Libération mémoire JVM
del df_train_original
spark.catalog.clearCache()

# --- 2/4 : Pitch Shift (append) ---
print("🔧 [2/4] Augmentation Pitch Shift (append)...")
df_aug_pitch = (
    df_train
    .withColumn("content", pitch_shift_udf(col("content")))
    .withColumn("augmentation", lit("pitch_shift"))
)
df_aug_pitch.write.mode("append").parquet(AUGMENTED_PARQUET)
print("   ✅ Pitch Shift sauvegardé")

# Libération mémoire JVM
del df_aug_pitch
spark.catalog.clearCache()

# --- 3/4 : Bruit Blanc (append) ---
print("🔧 [3/4] Augmentation Bruit Blanc (append)...")
df_aug_noise = (
    df_train
    .withColumn("content", add_noise_udf(col("content")))
    .withColumn("augmentation", lit("white_noise"))
)
df_aug_noise.write.mode("append").parquet(AUGMENTED_PARQUET)
print("   ✅ Bruit Blanc sauvegardé")

# Libération mémoire JVM
del df_aug_noise
spark.catalog.clearCache()

# --- 4/4 : Time Stretch (append) ---
print("🔧 [4/4] Augmentation Time Stretch (append)...")
df_aug_stretch = (
    df_train
    .withColumn("content", time_stretch_udf(col("content")))
    .withColumn("augmentation", lit("time_stretch"))
)
df_aug_stretch.write.mode("append").parquet(AUGMENTED_PARQUET)
print("   ✅ Time Stretch sauvegardé")

# Libération mémoire JVM finale
del df_aug_stretch
spark.catalog.clearCache()

# --- Relecture depuis le disque — la RAM JVM est désormais libre ---
print("\n📖 Relecture du Parquet consolidé...")
df_train_augmented = spark.read.parquet(AUGMENTED_PARQUET)

augmented_count = df_train_augmented.count()
print(f"📈 Taille du train augmenté : {augmented_count} (~4x {train_count})")
df_train_augmented.groupBy("augmentation").count().orderBy("augmentation").show()

🔧 [1/4] Écriture des originaux (overwrite)...


   ✅ Originaux sauvegardés (9680 fichiers)
🔧 [2/4] Augmentation Pitch Shift (append)...


   ✅ Pitch Shift sauvegardé
🔧 [3/4] Augmentation Bruit Blanc (append)...


   ✅ Bruit Blanc sauvegardé
🔧 [4/4] Augmentation Time Stretch (append)...


   ✅ Time Stretch sauvegardé

📖 Relecture du Parquet consolidé...


📈 Taille du train augmenté : 38720 (~4x 9680)


+------------+-----+
|augmentation|count|
+------------+-----+
|    original| 9680|
| pitch_shift| 9680|
|time_stretch| 9680|
| white_noise| 9680|
+------------+-----+



---
## 🎵 Étape 6 — Extraction de Features Multi-Échelle (Style Babko)

UDF complexe qui extrait un **vecteur de 185 dimensions** (9 caractéristiques) :

| # | Feature | Dimensions | Librairie |
|---|---------|------------|----------|
| 1 | MFCC (40 coefficients) | 40 | `librosa.feature.mfcc` |
| 2 | Zero Crossing Rate | 1 | `librosa.feature.zero_crossing_rate` |
| 3 | RMS Energy | 1 | `librosa.feature.rms` |
| 4 | Spectral Centroid | 1 | `librosa.feature.spectral_centroid` |
| 5 | Spectral Rolloff | 1 | `librosa.feature.spectral_rolloff` |
| 6 | Chroma STFT (12 bins) | 12 | `librosa.feature.chroma_stft` |
| 7 | Mel Spectrogram (128 bins) | 128 | `librosa.feature.melspectrogram` |
| 8 | Entropy of Energy | 1 | Calcul manuel (Shannon) |
| | **Total** | **185** | |

In [8]:
@udf(returnType=ArrayType(FloatType()))
def extract_features_udf(audio_bytes):
    """
    Extrait un vecteur de 185 features (9 caractéristiques Babko)
    à partir de données audio binaires WAV.

    Composition du vecteur :
        [0:40]    MFCC           — 40 coefficients
        [40]      ZCR            — 1 valeur
        [41]      RMS Energy     — 1 valeur
        [42]      Spectral Centr — 1 valeur
        [43]      Spectral Roll  — 1 valeur
        [44:56]   Chroma STFT    — 12 valeurs
        [56:184]  Mel Spectro    — 128 valeurs
        [184]     Entropy Energy — 1 valeur
    """
    import io, librosa, numpy as np
    try:
        # Décodage binaire → signal numpy (sr hardcodé à 22050)
        y, sr = librosa.load(io.BytesIO(audio_bytes), sr=22050)

        # 1. MFCC — 40 coefficients (moyenne temporelle)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        mfcc_mean = np.mean(mfcc, axis=1)  # (40,)

        # 2. Zero Crossing Rate (moyenne temporelle)
        zcr = librosa.feature.zero_crossing_rate(y=y)
        zcr_mean = np.mean(zcr)

        # 3. RMS Energy (moyenne temporelle)
        rms = librosa.feature.rms(y=y)
        rms_mean = np.mean(rms)

        # 4. Spectral Centroid (moyenne temporelle)
        sc = librosa.feature.spectral_centroid(y=y, sr=sr)
        sc_mean = np.mean(sc)

        # 5. Spectral Rolloff (moyenne temporelle)
        sro = librosa.feature.spectral_rolloff(y=y, sr=sr)
        sro_mean = np.mean(sro)

        # 6. Chroma STFT — 12 bins (moyenne temporelle)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = np.mean(chroma, axis=1)  # (12,)

        # 7. Mel Spectrogram — 128 bins (moyenne temporelle)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel_mean = np.mean(mel, axis=1)  # (128,)

        # 8. Entropy of Energy (entropie de Shannon)
        # Découpage en frames, calcul de l'énergie par frame,
        # normalisation en distribution de probabilité, puis entropie
        frame_length = 2048
        hop_length = 512
        frames = librosa.util.frame(y, frame_length=frame_length, hop_length=hop_length)
        energy = np.sum(frames ** 2, axis=0)
        energy_sum = np.sum(energy)
        if energy_sum > 0:
            prob = energy / energy_sum
            # Éviter log(0) en filtrant les valeurs nulles
            prob = prob[prob > 0]
            entropy = -np.sum(prob * np.log2(prob))
        else:
            entropy = 0.0

        # Concaténation → vecteur de 185 dimensions
        features = np.concatenate([
            mfcc_mean,           # 40
            [zcr_mean],          # 1
            [rms_mean],          # 1
            [sc_mean],           # 1
            [sro_mean],          # 1
            chroma_mean,         # 12
            mel_mean,            # 128
            [entropy]            # 1
        ])

        return [float(f) for f in features]

    except Exception:
        # En cas d'erreur de décodage, retourner un vecteur nul
        return [0.0] * 185

In [9]:
# --- Application sur le TRAIN (augmenté) ---
print("🎵 Extraction des features du set TRAIN...")
df_train_features = (
    df_train_augmented
    .withColumn("features", extract_features_udf(col("content")))
    .select("path", "label", "augmentation", "features")
)

# --- Application sur le TEST ---
print("🎵 Extraction des features du set TEST...")
df_test_features = (
    df_test
    .withColumn("augmentation", lit("original"))
    .withColumn("features", extract_features_udf(col("content")))
    .select("path", "label", "augmentation", "features")
)

# Aperçu rapide (on ne fait PAS de .count() ici pour éviter de tout matérialiser)
print("\n📊 Aperçu du vecteur de features (1er échantillon train) :")
sample_row = df_train_features.select("label", "features").first()
if sample_row:
    print(f"   Label : {sample_row['label']}")
    print(f"   Features[:10] : {sample_row['features'][:10]}")
    print(f"   Dimension du vecteur : {len(sample_row['features'])}")

🎵 Extraction des features du set TRAIN...
🎵 Extraction des features du set TEST...

📊 Aperçu du vecteur de features (1er échantillon train) :


   Label : angry
   Features[:10] : [-274.1614074707031, 13.640148162841797, -9.983597755432129, -1.0225878953933716, -7.311028957366943, -4.149641990661621, 0.4161698520183563, -5.810477256774902, -4.710836887359619, -2.898430824279785]
   Dimension du vecteur : 185


---
## 💾 Étape 7 — Sauvegarde en Apache Parquet

> ⚠️ **Pas de `.cache()` ni `.count()` avant l'écriture** — on laisse Spark
> écrire directement en streaming pour éviter le OutOfMemoryError.
>
> La vérification se fait **après** la relecture du Parquet.

In [10]:
PARQUET_BASE = "/mnt/c/Users/joane/Desktop/ESGI4/Spark Core/voice-rec/"

train_parquet_path = os.path.join(PARQUET_BASE, "train_features.parquet")
test_parquet_path = os.path.join(PARQUET_BASE, "test_features.parquet")

# --- Sauvegarde directe (pas de .cache() / .count() avant !) ---
print(f"💾 Sauvegarde train → {train_parquet_path}")
df_train_features.write.mode("overwrite").parquet(train_parquet_path)

print(f"💾 Sauvegarde test  → {test_parquet_path}")
df_test_features.write.mode("overwrite").parquet(test_parquet_path)

# --- Vérification APRÈS écriture (relecture légère) ---
df_check_train = spark.read.parquet(train_parquet_path)
df_check_test = spark.read.parquet(test_parquet_path)

print(f"\n✅ Train Parquet sauvegardé — {df_check_train.count()} lignes")
print(f"✅ Test  Parquet sauvegardé — {df_check_test.count()} lignes")
print(f"   Colonnes : {df_check_train.columns}")

print("\n📊 Distribution des labels (train) :")
df_check_train.groupBy("label").count().orderBy("count", ascending=False).show()

print("📊 Distribution des augmentations (train) :")
df_check_train.groupBy("augmentation").count().orderBy("count", ascending=False).show()

💾 Sauvegarde train → /mnt/c/Users/joane/Desktop/ESGI4/Spark Core/voice-rec/train_features.parquet


/home/darky/miniconda3/envs/voice_rec/lib/python3.14/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
/home/darky/miniconda3/envs/voice_rec/lib/python3.14/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
/home/darky/miniconda3/envs/voice_rec/lib/python3.14/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


💾 Sauvegarde test  → /mnt/c/Users/joane/Desktop/ESGI4/Spark Core/voice-rec/test_features.parquet



✅ Train Parquet sauvegardé — 38720 lignes


✅ Test  Parquet sauvegardé — 2482 lignes
   Colonnes : ['path', 'label', 'augmentation', 'features']

📊 Distribution des labels (train) :


+--------+-----+
|   label|count|
+--------+-----+
|     sad| 6248|
| disgust| 6220|
|    fear| 6204|
|   happy| 6108|
|   angry| 6072|
| neutral| 5812|
|      ps| 1236|
|surprise|  820|
+--------+-----+

📊 Distribution des augmentations (train) :
+------------+-----+
|augmentation|count|
+------------+-----+
| white_noise| 9680|
|    original| 9680|
| pitch_shift| 9680|
|time_stretch| 9680|
+------------+-----+



---
## 🔊 Étape 8 — Vérification Audio (Original vs Augmenté)

Écoute d'un échantillon audio directement dans le notebook pour vérifier que
les augmentations n'ont pas corrompu le signal.

In [11]:
from IPython.display import Audio, display

# --- Relecture des données augmentées depuis le Parquet intermédiaire ---
df_aug_check = spark.read.parquet(AUGMENTED_PARQUET)

# --- 🔊 Écouter un échantillon ORIGINAL ---
sample_orig = (
    df_aug_check
    .filter(col("augmentation") == "original")
    .select("content", "label")
    .first()
)
if sample_orig:
    y_orig, sr_orig = librosa.load(io.BytesIO(sample_orig["content"]), sr=SAMPLE_RATE)
    print(f"🔊 Échantillon ORIGINAL — Label : '{sample_orig['label']}', "
          f"Durée : {len(y_orig)/sr_orig:.2f}s, SR : {sr_orig}Hz")
    display(Audio(y_orig, rate=sr_orig))

🔊 Échantillon ORIGINAL — Label : 'angry', Durée : 4.00s, SR : 22050Hz


In [12]:
# --- 🔊 Écouter le MÊME échantillon en version augmentée (Pitch Shift) ---
sample_pitch = (
    df_aug_check
    .filter(col("augmentation") == "pitch_shift")
    .select("content", "label")
    .first()
)
if sample_pitch:
    y_pitch, sr_pitch = librosa.load(io.BytesIO(sample_pitch["content"]), sr=SAMPLE_RATE)
    print(f"🔊 Échantillon PITCH SHIFT — Label : '{sample_pitch['label']}', "
          f"Durée : {len(y_pitch)/sr_pitch:.2f}s")
    display(Audio(y_pitch, rate=sr_pitch))

🔊 Échantillon PITCH SHIFT — Label : 'angry', Durée : 4.00s


In [13]:
# --- 🔊 Écouter en version augmentée (Bruit Blanc) ---
sample_noise = (
    df_aug_check
    .filter(col("augmentation") == "white_noise")
    .select("content", "label")
    .first()
)
if sample_noise:
    y_noise, sr_noise = librosa.load(io.BytesIO(sample_noise["content"]), sr=SAMPLE_RATE)
    print(f"🔊 Échantillon BRUIT BLANC — Label : '{sample_noise['label']}', "
          f"Durée : {len(y_noise)/sr_noise:.2f}s")
    display(Audio(y_noise, rate=sr_noise))

🔊 Échantillon BRUIT BLANC — Label : 'angry', Durée : 4.00s


In [14]:
# --- 🔊 Écouter en version augmentée (Time Stretch) ---
sample_stretch = (
    df_aug_check
    .filter(col("augmentation") == "time_stretch")
    .select("content", "label")
    .first()
)
if sample_stretch:
    y_stretch, sr_stretch = librosa.load(io.BytesIO(sample_stretch["content"]), sr=SAMPLE_RATE)
    print(f"🔊 Échantillon TIME STRETCH — Label : '{sample_stretch['label']}', "
          f"Durée : {len(y_stretch)/sr_stretch:.2f}s")
    display(Audio(y_stretch, rate=sr_stretch))

🔊 Échantillon TIME STRETCH — Label : 'angry', Durée : 3.34s


---
## 🏁 Résumé du Pipeline

In [15]:
# Relecture finale pour le résumé
final_train = spark.read.parquet(train_parquet_path)
final_test = spark.read.parquet(test_parquet_path)

print("=" * 65)
print("🏁 Pipeline SER terminé avec succès !")
print("=" * 65)
print(f"   Échantillons train (avec augmentation) : {final_train.count()}")
print(f"   Échantillons test :                      {final_test.count()}")
print(f"   Dimension du vecteur de features :       185")
print(f"   Sortie Parquet :                         {PARQUET_BASE}")
print("=" * 65)
print()
print("📋 Détail du vecteur de 185 dimensions :")
print("   [0:40]    MFCC (40 coefficients)")
print("   [40]      Zero Crossing Rate")
print("   [41]      RMS Energy")
print("   [42]      Spectral Centroid")
print("   [43]      Spectral Rolloff")
print("   [44:56]   Chroma STFT (12 bins)")
print("   [56:184]  Mel Spectrogram (128 bins)")
print("   [184]     Entropy of Energy")
print("=" * 65)
print()
print("✅ Anti-leakage : split AVANT augmentation")
print("✅ Anti-OOM : micro-partitionnement + persistance itérative")
print("✅ UDFs sûres : sr=22050 hardcodé (pas de closure)")

🏁 Pipeline SER terminé avec succès !
   Échantillons train (avec augmentation) : 38720


   Échantillons test :                      2482
   Dimension du vecteur de features :       185
   Sortie Parquet :                         /mnt/c/Users/joane/Desktop/ESGI4/Spark Core/voice-rec/

📋 Détail du vecteur de 185 dimensions :
   [0:40]    MFCC (40 coefficients)
   [40]      Zero Crossing Rate
   [41]      RMS Energy
   [42]      Spectral Centroid
   [43]      Spectral Rolloff
   [44:56]   Chroma STFT (12 bins)
   [56:184]  Mel Spectrogram (128 bins)
   [184]     Entropy of Energy

✅ Anti-leakage : split AVANT augmentation
✅ Anti-OOM : micro-partitionnement + persistance itérative
✅ UDFs sûres : sr=22050 hardcodé (pas de closure)
